In [1]:
import sys
# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.utils import Logger

/home/dzigen/Desktop/PersonalAI/pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


#### 1. Задаём конфигурацию графа знаний

In [3]:
# CONFIG FOR REMOTE STORAGE

remote_kg_config = RemoteKnowledgeGraphConfig(
    graph_sturct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='neo4j', db_config=GraphDBConnectionConfig(uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password', 'db_name': 'testing'}))),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing', db_name='vectorized_nodes', is_exist=False, need_to_clear=False)), 
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing', db_name='vectorized_triplets', is_exist=False, need_to_clear=False)),
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small')),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar', # TO CHANGE
            retriever_config=AStarGraphSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='aerospike', db_config=KVDBConnectionConfig(host='aerospikelservice', port=3000))),
        answer_generator_config=QALLMGeneratorConfig()),
    log=Logger('main_debug')
)

In [3]:
# CONFIG FOR IN-MEMORY STORAGE

# GraphModelConfig(driver_config=GraphDriverConfig(db_vendor='inmemory_graph', db_config=GraphDBConnectionConfig(uri='')))

inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='neo4j', db_config=GraphDBConnectionConfig(uri="bolt://localhost:7687", params={'user': "neo4j", 'pwd': 'password', 'db_name': 'testing'}))),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing6', db_name='vectorized_nodes', is_exist=True, need_to_clear=True)), 
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing6', db_name='vectorized_triplets', is_exist=True, need_to_clear=True)),
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small')),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='bfs', # TO CHANGE
            retriever_config=BFSSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=KVDBConnectionConfig(
                host='localhost', params={'kvstore_dump_name': 'inmemory_store', 'load_from_disk': False, 'load_dump_dir': '.', # TO CHANGE
                                          'save_on_disk': True, 'save_dump_dir': '.'}))), # TO CHANGE
        answer_generator_config=QALLMGeneratorConfig()),
    log=Logger('main_debug'))

#### 2. Инициализируем граф знаний

In [4]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [ ]:
# ATTENTION !!!
rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 3. Добавляем в граф информацию

In [ ]:
rkg_main.update_memory([
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'",
    "Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University",
    "Petrozavodsk State University is where Mikhail Menshchikov received his bachelor's degree.",
    "Mikhail Menshchikov studied at Petrozavodsk State University and received a bachelor's degree."])

#### 4. Q&A

In [ ]:
rkg_main.answer_question("What program is Mikhail Menshchikov studying for his master's degree?")

In [ ]:
rkg_main.answer_question("Where did Mikhail Menshchikov receive his bachelor's degree?")